# Format and compare processed data

This notebook takes the processed data from the DADA2 and VSEARCH pipelines and explores their differences. 
- In total, nine comparisons including 3 denoising methods (DADA2, VSEARCH and MetaBEAT) x 3 taxonomy assignment methods (RDP, BLAST LCA and MAPseq) 
- Comparisons are repeated across three datasets which have known fish species from traditional surveying (Ring trial, Marchamley, and Windermere)
- Compares ASVs, sequences, and taxa (species, genus and family)
- Explores precision and accuracy with traditional data 

## Set-up

In [0]:
# Install packages
installed <- rownames(installed.packages())

ensure_cran <- function(pkgs, repos = "https://www.stats.bris.ac.uk/R/") {
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    install.packages(to_install, dependencies = TRUE, repos = repos)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

ensure_bioc <- function(pkgs) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
  }
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    BiocManager::install(to_install, ask = FALSE, update = FALSE)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

## CRAN packages
cran_pkgs <- c("Rcpp", "devtools", "ggplot2", "optparse", "taxonomizr", "dplyr", "seqinr", "worrms", "tidyverse", "janitor")
ensure_cran(cran_pkgs)

## Bioconductor packages
bioc_pkgs <- c("Biostrings", "ShortRead", "dada2", "phyloseq", "microbiome")
ensure_bioc(bioc_pkgs)

In [0]:
%python
pip install biopython

## Import and merge data


For each dataset (n = 3) and denoising option (n = 3), we have the following five processed files.

For ASV information before taxonomic assignment:
- ASV counts
- ASV seqs

For taxonomic assignment:
- RDP
- MAPseq 
- BLAST (LCA condensed)

We have lots of different files here which need to be manipulated into one single data frame. We can create both long and phyloseq formats. 

Note: MetaBEAT files are formatteed differently and have to be cleaned before introduced to the script.

### Create master long dataset

Let's import and merge all the data into one master long dataset.

In [0]:
%sh
python -u Scripts/10_create_master_dataset.py # parameters to change at top of script

In [0]:
# read in data in R
master_long_df <- read.csv("Data/Processed/master_long_data.csv")

# set factor variables
factor_cols <- c("asv","sampleid","denoise_method","dataset","taxonomy_method","kingdom","phylum","class","order","family","genus","species"
)

factor_cols <- factor_cols[factor_cols %in% colnames(master_long_df)]
master_long_df[factor_cols] <- lapply(master_long_df[factor_cols], as.factor)

# check expected variables are there
#str(master_long_df)
print(levels(master_long_df$taxonomy_method))
print(levels(master_long_df$denoise_method))
print(levels(master_long_df$dataset))

# create a unique fullID
master_long_df$fullID = paste(master_long_df$sampleid, master_long_df$denoise_method, master_long_df$taxonomy_method, master_long_df$dataset, sep="_")

In [0]:
str(master_long_df)

### Extracting information from WoRMS

We often get random spaces or phrases we don’t want in our taxonomic names. We can use the janitor package to clean our taxonomic names.

We can also get all metadata associated to our taxa from the World Register of Marine Species (WoRMS). 

This next script does both of those jobs and outputs a new long dataset called `master_long_worms_df`. It is also saved in the Data/Processed folder.

In [0]:
source("Scripts/10b_tidy_WoRMS.R")

In [0]:

str(master_long_worms_df)

### Create master phyloseq

In [0]:
source("Scripts/10c_create_phyloseq.R")

#### Check phyloseq object

In [0]:
phylo_eDNA

In [0]:
head(sample_names(phylo_eDNA)) #check fullid names

In [0]:
rank_names(phylo_eDNA) #check taxonomic information

In [0]:
sample_variables(phylo_eDNA) #check meta data 

In [0]:
microbiome::summarize_phyloseq(phylo_eDNA) #full summary

## Comparisons

### Explore unassigned ASVs/OTUs & taxa

In [0]:
source("Scripts/11_check_unassigned.R")

In [0]:
unassigned_summary_table

In [0]:
asv_data <- subset(unassigned_summary_table, metric == "ASV")

ggplot(asv_data,
       aes(x = denoise_method, y = percent_unassigned, group = dataset, color = dataset)) +
  geom_line() +
  geom_point() +
  facet_grid(level ~ taxonomy_method) +
  theme_bw() +
  labs(
    x = "Denoising tool",
    y = "Percent unassigned ASVs / OTUs (%)",
    color = "Dataset"
  )

In [0]:
ggplot(asv_data,
       aes(x = taxonomy_method, y = percent_unassigned, group = dataset, color = dataset)) +
  geom_line() +
  geom_point() +
  facet_grid(level ~ denoise_method) +
  theme_bw() +
  labs(
    x = "Taxonomic assignment tool",
    y = "Percent unassigned ASVs / OTUs (%)",
    color = "Dataset"
  )

In [0]:

taxa_data <- subset(unassigned_summary_table, metric == "taxa")

ggplot(taxa_data,
       aes(x = denoise_method, y = percent_unassigned, group = dataset, color = dataset)) +
  geom_line() +
  geom_point() +
  facet_grid( level ~ taxonomy_method) +
  theme_bw() +
  labs(
    x = "Denoising tool",
    y = "Percent unassigned taxa (%)",
    color = "Dataset"
  )


In [0]:
ggplot(taxa_data,
       aes(x = taxonomy_method, y = percent_unassigned, group = dataset, color = dataset)) +
  geom_line() +
  geom_point() +
  facet_grid( level ~ denoise_method) +
  theme_bw() +
  labs(
    x = "Taxonomic assignment tool",
    y = "Percent unassigned taxa (%)",
    color = "Dataset"
  )

### Explore taxonomy

Let's make some broad family stacked plots across all methods and datasets.

In [0]:
ps_march <- subset_samples(phylo_eDNA, dataset == "marchamley")
ps_march <- prune_taxa(taxa_sums(ps_march) > 0, ps_march)
ps_family <- tax_glom(ps_march, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_march <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Marchamley)") +
  ggplot2::theme_bw()+
  theme(legend.position = "none",
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_march)

ggsave(
  filename = "Results/Marchamley/Marchamley_Family_barplot.png",
  plot = p_march,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

In [0]:
ps_wind <- subset_samples(phylo_eDNA, dataset == "windermere_2017")
ps_wind <- prune_taxa(taxa_sums(ps_wind) > 0, ps_wind)
ps_family <- tax_glom(ps_wind, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_wind <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Winderemere)") +
  ggplot2::theme_bw()+
  theme(legend.position = "none",
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_wind)

ggsave(
  filename = "Results/Windermere_2017/Winderemere_Family_barplot.png",
  plot = p_wind,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

In [0]:
ps_ring <- subset_samples(phylo_eDNA, dataset == "ringtrial_sean")
ps_ring <- prune_taxa(taxa_sums(ps_ring) > 0, ps_ring)
ps_family <- tax_glom(ps_ring, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_ring <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Ring Trial)") +
  ggplot2::theme_bw()+
  theme(legend.position = "none",
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_ring)

ggsave(
  filename = "Results/RingTrial_Sean/RingTrial_Family_barplot.png",
  plot = p_ring,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

### Explore diversity

In [0]:
alpha_div <- estimate_richness(
  phylo_eDNA,
  measures = c("Observed", "Shannon")
)

meta <- data.frame(sample_data(phylo_eDNA))
df <- cbind(alpha_div, meta)

p_richness <- ggplot(df, aes(x = denoise_method, y = Observed, fill = denoise_method)) +
  geom_boxplot() +
  facet_grid(dataset ~ taxonomy_method) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = "Denoising Method",
       y = "Richness Diversity")

p_richness

In [0]:
ggplot(df, aes(x = denoise_method, y = Observed, fill = taxonomy_method)) +
  geom_boxplot() +
  facet_wrap(~ dataset) +
  theme_bw()


In [0]:
ggplot(df, aes(x = taxonomy_method, y = Observed, fill = denoise_method)) +
  geom_boxplot() +
  facet_wrap(~ dataset) +
  theme_bw()